In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate

from src.custom_functions import get_parameters_files, histogram_marginal, load_histogram, MSE, RMSE, histogram_CI_threshold, convex_hull_construction
from src.data_loader import load_data
import pandas as pd
import glob, pdb

densities = np.arange(10,22,2,dtype=int)  
x_d = np.arange(75, 1875.1, 50)
x_m = np.arange(62.5, 1900,  25)

fontsize=18
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

## Parameter estimates

In [2]:
parameter_names = "rm_rp" 
data_type = "artificial"
pred_type = "fit_all"
simulations = range(4)


for estimate_type in ["mean", "MAP"]:
    
    table = []
    table.append([""] + len(simulations)*["True parameters, (Rel. Error)"])

    table_tmp = ["True"]
    # ------------------------------------------
    # STEP 1: PARSE GROUND-TRUTH PARAMETERS
    # ------------------------------------------
    for simulation in simulations:

        parameter_names = "rm_rp"
        
        input_parameters = [None, parameter_names, "PDE", data_type, simulation, pred_type]

        (parameter_names, 
        model_type, 
        data_type, 
        dens, 
        simulation,
        initial_perc,
        trueParams,
        delay, 
        lower_bounds,
        upper_bounds,
        indepParams,
        dxs,
        prediction_type,
        priorDir, 
        histogramDir, 
        posteriorDir) = get_parameters_files(input_parameters)

        trueParams = np.array(trueParams)
        ### convert r_m from ABM units^2/day to mm^2/day
        trueParams[0] *= 625/1e6
        formatted_str = f"({', '.join(f'{x:.3f}' for x in trueParams)})$^T$"
        table_tmp.append(formatted_str)

    table.append(table_tmp)

    # ------------------------------------------
    # STEP 2: PARSE PARAMETER ESTIMATES (PDE vs ABM)
    # ------------------------------------------
    headers = [" "]
    for i in range(1,5):
        headers.append("\multicolumn{1}{c}{Dataset " + str(i) + "}")
    
    for model_type in ["PDE","ABM"]:

        table_tmp =  [model_type]

        for simulation in simulations:    

            parameter_names = "rm_rp"
            
            input_parameters = [None, parameter_names, model_type, data_type, simulation,pred_type]

            (parameter_names, 
            model_type, 
            data_type, 
            dens, 
            simulation,
            initial_perc,
            trueParams,
            delay, 
            lower_bounds,
            upper_bounds,
            indepParams,
            dxs,
            prediction_type,
            priorDir, 
            histogramDir, 
            posteriorDir) = get_parameters_files(input_parameters)

            
            trueParams = np.array(trueParams)

            #extract parameter estimate (mean or MAP)
            mat = np.load(f"{posteriorDir}mean_info.npy",allow_pickle=True).item()
            estimate = mat[estimate_type]
            estimate = np.array([estimate[0],estimate[2]])
            
            trueParamsDiv = trueParams
            trueParamsDiv[trueParams==0] = 1
            Err = np.linalg.norm( (trueParams-estimate)/trueParams ) 
            ### convert r_m from ABM units^2/day to mm^2/day
            estimate[0] *= 625/1e6
            
            formatted_str = f"({', '.join(f'{x:.3f}' for x in estimate)})$^T$"
            formatted_str += f", [{Err:.3f}]"
            
            table_tmp.append(formatted_str)

        table.append(table_tmp)

    ### convert table to latex format, do some cleaning afterwards
    table_df = pd.DataFrame(data=table[1:],columns=headers)

    latex_table = table_df.to_latex(
        index=False,  # To not include the DataFrame index as a column in the table
        position="h",  # The preferred positions where the table should be placed in the document ('here', 'top', 'bottom', 'page')
        column_format="|c|llll|",  # The format of the columns: left-aligend first column and center-aligned remaining columns as per APA guidelines
        escape=False,  # Disable escaping LaTeX special characters in the DataFrame
        float_format="{:0.2f}".format  # Formats floats to two decimal places
    )

    cleaned_table = (
        latex_table.replace(r"\toprule", r"\hline")
        .replace(r"\midrule", r"\hline")
        .replace(r"\bottomrule", r"\hline")
    )

    print(cleaned_table) 

\begin{table}[h]
\begin{tabular}{|c|llll|}
\hline
  & \multicolumn{1}{c}{Dataset 1} & \multicolumn{1}{c}{Dataset 2} & \multicolumn{1}{c}{Dataset 3} & \multicolumn{1}{c}{Dataset 4} \\
\hline
True & (0.031, 0.500)$^T$ & (0.031, 2.000)$^T$ & (0.156, 0.500)$^T$ & (0.156, 2.000)$^T$ \\
PDE & (0.033, 0.488)$^T$, [0.065] & (0.034, 1.923)$^T$, [0.098] & (0.146, 0.501)$^T$, [0.065] & (0.169, 1.956)$^T$, [0.084] \\
ABM & (0.036, 0.489)$^T$, [0.156] & (0.033, 1.998)$^T$, [0.071] & (0.150, 0.503)$^T$, [0.038] & (0.164, 2.001)$^T$, [0.051] \\
\hline
\end{tabular}
\end{table}

\begin{table}[h]
\begin{tabular}{|c|llll|}
\hline
  & \multicolumn{1}{c}{Dataset 1} & \multicolumn{1}{c}{Dataset 2} & \multicolumn{1}{c}{Dataset 3} & \multicolumn{1}{c}{Dataset 4} \\
\hline
True & (0.031, 0.500)$^T$ & (0.031, 2.000)$^T$ & (0.156, 0.500)$^T$ & (0.156, 2.000)$^T$ \\
PDE & (0.035, 0.510)$^T$, [0.127] & (0.034, 1.904)$^T$, [0.095] & (0.133, 0.464)$^T$, [0.167] & (0.168, 1.942)$^T$, [0.079] \\
ABM & (0.018, 0.466)$

## Parameter estimation for simulations 5-8

In [3]:
data_type = "artificial"
pred_type = "fit_all"
simulations = np.arange(4,8)

for estimate_type in ["mean", "MAP"]:
    
    table = []
    table.append([""] + len(simulations)*["True parameters, (Rel. Error)"])

    table_tmp = ["True"]
    # ------------------------------------------
    # STEP 1: PARSE GROUND-TRUTH PARAMETERS
    # ------------------------------------------
    for simulation in simulations:

        if simulation < 4: #Model 1
            parameter_names = "rm_rp"
        elif simulation == 4: #Model 2 
            parameter_names = "rm_pint_rp"
        elif simulation in [5,6]: #Model 3
            parameter_names = "rm_rp_a0_a1"
        elif simulation == 7: #Model 4
            parameter_names = "rm_pint_rp_a0_a1"
        
        input_parameters = [None, parameter_names, "PDE", data_type, simulation, pred_type]

        (parameter_names, 
        model_type, 
        data_type, 
        dens, 
        simulation,
        initial_perc,
        trueParams,
        delay, 
        lower_bounds,
        upper_bounds,
        indepParams,
        dxs,
        prediction_type,
        priorDir, 
        histogramDir, 
        posteriorDir) = get_parameters_files(input_parameters)

        trueParams = np.array(trueParams)
        ### convert r_m from ABM units^2/day to mm^2/day
        trueParams[0] *= 625/1e6
        formatted_str = f"({', '.join(f'{x:.3f}' for x in trueParams)})$^T$"
        table_tmp.append(formatted_str)

    table.append(table_tmp)
    
    headers = [" "]
    for i in range(5,9):
        headers.append("\multicolumn{1}{c}{Dataset " + str(i) + "}")
    # ------------------------------------------
    # STEP 2: PARSE PARAMETER ESTIMATES (PDE vs ABM)
    # ------------------------------------------
    for model_type in ["PDE","ABM"]:

        table_tmp =  [model_type ]

        for simulation in simulations:    

            if simulation < 4: #Model 1
                parameter_names = "rm_rp"
            elif simulation == 4: #Model 2 
                parameter_names = "rm_pint_rp"
            elif simulation in [5,6]: #Model 3
                parameter_names = "rm_rp_a0_a1"
            elif simulation == 7: #Model 4
                parameter_names = "rm_pint_rp_a0_a1"
            
            input_parameters = [None, parameter_names, model_type, data_type, simulation,pred_type]

            (parameter_names, 
            model_type, 
            data_type, 
            dens, 
            simulation,
            initial_perc,
            trueParams,
            delay, 
            lower_bounds,
            upper_bounds,
            indepParams,
            dxs,
            prediction_type,
            priorDir, 
            histogramDir, 
            posteriorDir) = get_parameters_files(input_parameters)

            trueParams = np.array(trueParams)
            
            #extract parameter estimate (mean or MAP)
            mat = np.load(f"{posteriorDir}mean_info.npy",allow_pickle=True).item()
            estimate = np.array(mat[estimate_type])
            
            if parameter_names == "rm_rp":
                estimate = np.array([estimate[0],estimate[2]])
            elif parameter_names == "rm_pint_rp":
                estimate = np.array([estimate[0],estimate[1],estimate[2]])
            elif parameter_names == "rm_rp_a0_a1":
                estimate = np.array([estimate[0],estimate[2],estimate[3],estimate[4]])
            elif parameter_names == "rm_pint_rp_a0_a1":
                pass

            trueParamsDiv = trueParams
            trueParamsDiv[trueParams==0] = 1
            Err = np.linalg.norm( (trueParams-estimate)/trueParams)
            
            ### convert r_m from ABM units^2/day to mm^2/day
            estimate[0] *= 625/1e6
            formatted_str = f"({', '.join(f'{x:.3f}' for x in estimate)})$^T$"
            formatted_str += f", [{Err:.3f}]"
            
            table_tmp.append(formatted_str)

        table.append(table_tmp)
    
    ### convert table to latex format, do some cleaning afterwards
    table_df = pd.DataFrame(data=table[1:],columns=headers)
    table_df.head()

    latex_table = table_df.to_latex(
        index=False,  # To not include the DataFrame index as a column in the table
        position="h",  # The preferred positions where the table should be placed in the document ('here', 'top', 'bottom', 'page')
        column_format="|c|llll|",  # The format of the columns: left-aligend first column and center-aligned remaining columns as per APA guidelines
        escape=False,  # Disable escaping LaTeX special characters in the DataFrame
        float_format="{:0.2f}".format  # Formats floats to two decimal places
    )

    cleaned_table = (
        latex_table.replace(r"\toprule", r"\hline")
        .replace(r"\midrule", r"\hline")
        .replace(r"\bottomrule", r"\hline")
    )

    print(cleaned_table)     

\begin{table}[h]
\begin{tabular}{|c|llll|}
\hline
  & \multicolumn{1}{c}{Dataset 5} & \multicolumn{1}{c}{Dataset 6} & \multicolumn{1}{c}{Dataset 7} & \multicolumn{1}{c}{Dataset 8} \\
\hline
True & (0.031, 0.500, 0.500)$^T$ & (0.156, 2.500, 0.000, 5.000)$^T$ & (0.156, 2.500, -1.250, 5.000)$^T$ & (0.156, 0.500, 2.500, 0.000, 5.000)$^T$ \\
PDE & (0.033, 0.556, 0.475)$^T$, [0.138] & (0.166, 2.449, -4.167, 26.667)$^T$, [6.744] & (0.163, 2.386, -4.000, 20.000)$^T$, [3.721] & (0.172, 0.417, 2.325, -3.200, 25.833)$^T$, [5.920] \\
ABM & (0.034, 0.373, 0.505)$^T$, [0.271] & (0.151, 2.536, -1.778, 20.000)$^T$, [4.089] & (0.162, 2.317, -3.075, 22.960)$^T$, [3.878] & (0.152, 0.587, 2.512, 0.462, 28.615)$^T$, [4.757] \\
\hline
\end{tabular}
\end{table}

\begin{table}[h]
\begin{tabular}{|c|llll|}
\hline
  & \multicolumn{1}{c}{Dataset 5} & \multicolumn{1}{c}{Dataset 6} & \multicolumn{1}{c}{Dataset 7} & \multicolumn{1}{c}{Dataset 8} \\
\hline
True & (0.031, 0.500, 0.500)$^T$ & (0.156, 2.500, 0.000, 5.0

In [4]:
parameter_names = "rm_pint_rp_a0_a1" 
data_type = "WH"
pred_type = "fit_all"
density_ids = range(6)
densities = np.arange(10,22,2)
model_type = "PDE"

headers = ["Dataset","\multicolumn{1}{|c|}{$\phat=(R_m,\ p_{pull},\ R_p,\ a_0,\ a_1)^T$}"]

for estimate_type in ["mean","MAP"]:

    table = []
    # ------------------------------------------
    # PARSE PARAMETER ESTIMATES (PDE vs ABM)
    # ------------------------------------------  
    for dens_id in density_ids:    

        input_parameters = [None, parameter_names, model_type, data_type, dens_id, pred_type]

        (parameter_names, 
        model_type, 
        data_type, 
        dens, 
        simulation,
        initial_perc,
        trueParams,
        delay, 
        lower_bounds,
        upper_bounds,
        indepParams,
        dxs,
        prediction_type,
        priorDir, 
        histogramDir, 
        posteriorDir) = get_parameters_files(input_parameters)

        if model_type == "ABM":posteriorDir = posteriorDir[:-1] + "_individual_testing/"

        #load in estimate (mean or MAP)
        mat = np.load(f"{posteriorDir}mean_info.npy",allow_pickle=True).item()
        estimate = mat[estimate_type]
        estimate = np.array(estimate)#[estimate[0],estimate[2]])
        
        ### convert r_m from ABM units^2/day to mm^2/day
        estimate[0] *= 625/1e6
        formatted_str = f"({', '.join(f'{x:.3f}' for x in estimate)})$^T$"
        # formatted_str += f", [{Err:.3f}]"

        table.append([f"Density {dens}K", formatted_str])


    ### convert table to latex format, do some cleaning afterwards
    table_df = pd.DataFrame(data=table,columns=headers)

    latex_table = table_df.to_latex(
        index=False,  # To not include the DataFrame index as a column in the table
        position="h",  # The preferred positions where the table should be placed in the document ('here', 'top', 'bottom', 'page')
        column_format="|c|l|",  # The format of the columns: left-aligend first column and center-aligned remaining columns as per APA guidelines
        escape=False,  # Disable escaping LaTeX special characters in the DataFrame
        float_format="{:0.2f}".format  # Formats floats to two decimal places
    )

    cleaned_table = (
        latex_table.replace(r"\toprule", r"\hline")
        .replace(r"\midrule", r"\hline")
        .replace(r"\bottomrule", r"\hline")
    )

    print(cleaned_table)     

\begin{table}[h]
\begin{tabular}{|c|l|}
\hline
Dataset & \multicolumn{1}{|c|}{$\phat=(R_m,\ p_{pull},\ R_p,\ a_0,\ a_1)^T$} \\
\hline
Density 10K & (0.033, 0.672, 1.321, -0.276, 19.080)$^T$ \\
Density 12K & (0.033, 0.500, 1.457, -11.333, 26.000)$^T$ \\
Density 14K & (0.072, 0.607, 1.460, -7.033, 21.905)$^T$ \\
Density 16K & (0.057, 0.681, 1.607, -8.889, 25.000)$^T$ \\
Density 18K & (0.072, 0.500, 1.786, -10.000, 25.000)$^T$ \\
Density 20K & (0.114, 0.467, 2.127, -10.174, 26.377)$^T$ \\
\hline
\end{tabular}
\end{table}

\begin{table}[h]
\begin{tabular}{|c|l|}
\hline
Dataset & \multicolumn{1}{|c|}{$\phat=(R_m,\ p_{pull},\ R_p,\ a_0,\ a_1)^T$} \\
\hline
Density 10K & (0.016, 0.300, 1.500, -16.000, 20.000)$^T$ \\
Density 12K & (0.028, 0.500, 1.196, -10.000, 30.000)$^T$ \\
Density 14K & (0.066, 0.750, 1.397, -9.231, 33.333)$^T$ \\
Density 16K & (0.036, 0.875, 1.607, -10.909, 25.000)$^T$ \\
Density 18K & (0.055, 0.500, 1.518, -7.500, 33.333)$^T$ \\
Density 20K & (0.098, 0.750, 2.071, -10.000

In [5]:
%load_ext watermark

%watermark -n -u -v -iv -w

Last updated: Fri Jun 05 2026

Python implementation: CPython
Python version       : 3.11.5
IPython version      : 8.15.0

pandas    : 2.1.4
numpy     : 1.25.2
matplotlib: 3.8.0

Watermark: 2.4.3

